# 02 · Predicting Suzuki–Miyaura yields with machine learning

In this notebook you train yield-prediction models on a published high-throughput Suzuki–Miyaura dataset, and learn how to **evaluate them**. Along the way you learn which reaction conditions matter, which helps you plan your own experiments.

**The dataset.** Perera *et al.* (*Science* **2018**, 359, 429) ran 5760 Suzuki–Miyaura couplings in a nanomole-scale flow platform. They varied:

| Component | Options |
|---|---|
| Quinoline coupling partner | 6-Cl-, 6-Br-, 6-I-, 6-OTf-quinoline (electrophiles), or 6-B(OH)₂-, 6-BPin-, 6-BF₃K-quinoline (nucleophiles) |
| Other partner | an indazole boronic acid / pinacol ester / BF₃K salt, or the matching indazole bromide |
| Ligand (with Pd(OAc)₂) | 11 phosphines + no ligand |
| Base | 7 bases + no base |
| Solvent | MeCN, THF, DMF, MeOH (aqueous mixtures for some runs) |

The closest analogue to the project reaction is **6-bromoquinoline + aryl boronic acid**: an aryl bromide coupled with an aryl boronic acid.

**By the end you should be able to:**
1. Load and explore a reaction dataset, and see which variables control the yield.
2. Turn reactions into model inputs: one-hot encodings and molecular fingerprints.
3. Train a random forest and evaluate it with **random** and **out-of-sample** splits, and explain why they give different answers.
4. Train an existing deep-learning yield model (Chemprop) on reaction SMILES.

**Further reading:**
- Perera *et al.*, *Science* **2018**, 359, 429: the dataset.
- Ahneman *et al.*, *Science* **2018**, 360, 186, and the comment by Chuang & Keiser, *Science* **2018**, 362, eaat8603: why baselines and out-of-sample tests matter. Read both.
- Schwaller *et al.*, *Mach. Learn.: Sci. Technol.* **2021**, 2, 015016: "Yield-BERT", a transformer on reaction SMILES; it also uses this dataset.
- Heid & Green, *J. Chem. Inf. Model.* **2022**, 62, 2101, and Heid *et al.*, *J. Chem. Inf. Model.* **2024**, 64, 9: Chemprop for reactions, and the Chemprop package paper.
- Beker *et al.*, *J. Am. Chem. Soc.* **2022**, 144, 4819: a sobering look at predicting Suzuki conditions from literature data.

In [ ]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_absolute_error

DATA = Path("data"); DATA.mkdir(exist_ok=True)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

## 1. Load the data

We use two files from the [rxn_yields](https://github.com/rxn4chemistry/rxn_yields) repository:
- the original spreadsheet from the paper's supplementary information (reagent names, yields);
- the same reactions written as **reaction SMILES**, in the same row order.

In [ ]:
BASE_URL = "https://github.com/rxn4chemistry/rxn_yields/raw/master/data/Suzuki-Miyaura/"
FILES = {"suzuki_perera.xlsx": "aap9112_Data_File_S1.xlsx",
         "suzuki_perera_rxn.tsv": "random_splits/random_split_0.tsv"}
for local, remote in FILES.items():
    if not (DATA / local).exists():
        import urllib.request
        urllib.request.urlretrieve(BASE_URL + remote, DATA / local)


def load_suzuki():
    raw = pd.read_excel(DATA / "suzuki_perera.xlsx")
    rxn = pd.read_csv(DATA / "suzuki_perera_rxn.tsv", sep="\t", index_col=0).sort_index()
    assert np.allclose(raw["Product_Yield_PCT_Area_UV"], rxn["y"] * 100), "row order mismatch"
    return pd.DataFrame({
        "quinoline": raw["Reactant_1_Short_Hand"].str.split(", ").str[1],
        "partner":   raw["Reactant_2_Name"].str.split(", ").str[1],
        "ligand":    raw["Ligand_Short_Hand"].str.strip().fillna("None"),
        "base":      raw["Reagent_1_Short_Hand"].fillna("None"),
        "solvent":   raw["Solvent_1_Short_Hand"],
        # "~" joins Pd and its acetates in the original; RDKit needs "." for separate fragments
        "rxn":       rxn["rxn"].str.replace("~", ".").values,
        "yield":     raw["Product_Yield_PCT_Area_UV"],
    })


df = load_suzuki()
COMPONENTS = ["quinoline", "partner", "ligand", "base", "solvent"]
print(df.shape)
print(df[COMPONENTS].nunique())
df.head()

**Note on the yields.** The "yield" here is HPLC UV peak-area percent of product, not an isolated yield. Different ways of measuring yield are not interchangeable. Pick one method for your own experiments and stick to it.

## 2. Explore before you model

Always look at the data first. What is the yield distribution? Which variables obviously matter?

## Your turn

1. plot a yield distribution
2. Plot mean yield by quinoline
3. Plot mean yield by ligand

In [ ]:
# The reaction closest to the project: aryl bromide + aryl boronic acid
br = df[(df.quinoline == "6-Br-Q") & (df.partner == "Boronic Acid")]
print(f"{len(br)} conditions; mean yield {br['yield'].mean():.1f}%, best {br['yield'].max():.1f}%")

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
for ax, solvent in zip(axes, sorted(br.solvent.unique())):
    table = br[br.solvent == solvent].pivot_table(index="ligand", columns="base", values="yield")
    im = ax.imshow(table, cmap="viridis", vmin=0, vmax=100)
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=90)
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.set_title(solvent)
fig.colorbar(im, ax=axes, shrink=0.8, label="yield (%)")
fig.suptitle("6-bromoquinoline + boronic acid: yield for every ligand × base × solvent combination");

In [ ]:
# How do the aryl bromide and the aryl chloride compare under the same conditions?
pair = df[df.partner == "Boronic Acid"].pivot_table(index=["ligand", "base", "solvent"], columns="quinoline", values="yield")
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pair["6-Br-Q"], pair["6-Cl-Q"], s=8, alpha=0.5)
ax.plot([0, 100], [0, 100], "k--", lw=1)
ax.set(xlabel="yield with 6-Br-quinoline (%)", ylabel="yield with 6-Cl-quinoline (%)",
       title="Same conditions, different halide");

## 3. Featurisation: turning reactions into numbers

Two simple representations:

1. **One-hot encoding.** Each reagent is just a label. The model knows nothing about chemistry, but this is the baseline every other model must beat.
2. **Molecular fingerprints.** For each molecule in the reaction we compute a Morgan (circular) count fingerprint and add them up. This is a "bag of molecules". Reagents that share substructures (e.g. two biaryl phosphines) now look similar to the model, so in principle it can generalise to a reagent it has not seen.

Notebook 01 showed a third option: **quantum-chemical descriptors**. Notebook 03 uses those for the Bayesian optimisation.

In [ ]:
def featurise_onehot(df):
    return pd.get_dummies(df[COMPONENTS]).astype(float).reset_index(drop=True)


_fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
_fp_cache = {}
def mol_fp(smi):
    if smi not in _fp_cache:
        _fp_cache[smi] = _fpgen.GetCountFingerprintAsNumPy(Chem.MolFromSmiles(smi)).astype(float)
    return _fp_cache[smi]


def featurise_fingerprint(df):
    # Sum of Morgan count fingerprints over all reactant-side molecules (reagents included).
    X = np.vstack([sum(mol_fp(s) for s in r.split(">>")[0].split(".")) for r in df["rxn"]])
    X = X[:, X.std(axis=0) > 0]           # drop bits that never change in this dataset
    return pd.DataFrame(X)


X = {"one-hot": featurise_onehot(df), "fingerprint": featurise_fingerprint(df)}
y = df["yield"].to_numpy()
for k, v in X.items():
    print(f"{k:12s} {v.shape}")

## 4. Random forest, random split

We use a 70/30 random split. Random forests are a good first model for tabular chemistry data. They need little tuning, handle non-linear effects and interactions (e.g. "this ligand only works with that base"), and are fairly robust on small datasets.

In [ ]:
def fit_rf(Xtr, ytr, seed=0):
    return RandomForestRegressor(n_estimators=300, max_features=0.3, n_jobs=-1, random_state=seed).fit(Xtr, ytr)

idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3, random_state=0)
rows, preds = [], {}
for name, Xf in X.items():
    p = fit_rf(Xf.iloc[idx_train], y[idx_train]).predict(Xf.iloc[idx_test])
    preds[name] = p
    rows.append({"features": name, "R2": r2_score(y[idx_test], p), "MAE": mean_absolute_error(y[idx_test], p)})
pd.DataFrame(rows).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True)
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(y[idx_test], p, s=5, alpha=0.4)
    ax.plot([0, 100], [0, 100], "k--", lw=1)
    ax.set(title=f"{name}: R² = {r2_score(y[idx_test], p):.2f}", xlabel="observed yield (%)")
axes[0].set_ylabel("predicted yield (%)")
fig.tight_layout()

## 5. Out-of-sample tests: can the model handle a reagent it has never seen?

In a full-factorial dataset like this one, a random test set contains reactions whose ligand, base, solvent and substrates were all seen many times in training. The model can do well just by learning "SPhos is good, no base is bad". That is useful for interpolating within the screen, but it says nothing about **new** reagents, which is what you care about when moving to your own substrates.

Here we hold out one ligand (or one quinoline coupling partner) at a time, train on everything else, and predict the held-out group.

In [ ]:
def leave_one_group_out(Xf, y, groups):
    oof = np.zeros_like(y)
    for tr, te in LeaveOneGroupOut().split(Xf, y, groups):
        oof[te] = fit_rf(Xf.iloc[tr], y[tr]).predict(Xf.iloc[te])
    return oof

rows, oof_preds = [], {}
for held_out in ["ligand", "quinoline"]:
    for name, Xf in X.items():
        oof = leave_one_group_out(Xf, y, df[held_out].to_numpy())
        oof_preds[(held_out, name)] = oof
        rows.append({"held-out": held_out, "features": name,
                     "R2": r2_score(y, oof), "MAE": mean_absolute_error(y, oof)})
pd.DataFrame(rows).round(3)

In [ ]:
# Which held-out ligands are hardest to predict?
per_ligand = pd.DataFrame({
    name: pd.Series(np.abs(oof_preds[("ligand", name)] - y)).groupby(df["ligand"].values).mean()
    for name in X}).sort_values("fingerprint")
per_ligand.plot.barh(figsize=(6, 4.5), title="MAE when this ligand is left out of training")
plt.xlabel("MAE (% yield)");

### Your turn (Parts 1–5)

1. Explain in two sentences why the random-split R² is much higher than the out-of-sample R². Which number would you quote in a report, and for which use case?
2. One-hot features for an unseen ligand are all zero. What does the random forest effectively predict in that case? Is the fingerprint model doing better than that?
3. From the 6-Br-Q vs 6-Cl-Q plot: which conditions work for the bromide but fail for the chloride? Relate this to oxidative addition and ligand properties (electron-rich, bulky phosphines).
4. Train a model on **6-Br-Q + boronic acid only** (384 reactions) and look at `feature_importances_` with one-hot features. Which ligand, base and solvent matter most? Compare with the heat maps.
5. **Y-randomisation control:** shuffle `y` and retrain. What R² do you get?

## 6. An existing deep-learning model: Chemprop

[Chemprop](https://github.com/chemprop/chemprop) is a widely used, maintained message-passing neural network (MPNN). In **reaction mode** it reads reaction SMILES directly and builds a *condensed graph of reaction* (CGR) from reactants and products. You don't have to design any features.

We train it on the same 70/30 split as the random forest. A further 10% of the training set is held back as a validation set for early stopping. With 30 epochs this takes ~5–10 min on a laptop CPU. Use more epochs (e.g. 100) for a fair comparison.

In [ ]:
CP_DIR = RESULTS / "chemprop_suzuki"; CP_DIR.mkdir(exist_ok=True)
CHEMPROP = str(Path(sys.executable).parent / "chemprop")   # the chemprop CLI in this environment

cp = df[["rxn", "yield"]].copy()
split = np.full(len(cp), "train", dtype=object)
split[idx_test] = "test"
rng = np.random.default_rng(0)
split[rng.choice(idx_train, size=len(idx_train) // 10, replace=False)] = "val"
cp["split"] = split
cp.to_csv(CP_DIR / "random_split.csv", index=False)
cp.loc[cp.split == "test", ["rxn"]].to_csv(CP_DIR / "test.csv", index=False)


def run(cmd, log):
    with open(log, "w") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        print(open(log).read()[-3000:])
        raise RuntimeError(f"{cmd[1]} failed, see {log}")


run([CHEMPROP, "train", "--data-path", str(CP_DIR / "random_split.csv"),
     "--reaction-columns", "rxn", "--target-columns", "yield", "--splits-column", "split",
     "--task-type", "regression", "--epochs", "30", "--num-workers", "0",
     "--output-dir", str(CP_DIR / "model_random")], CP_DIR / "train.log")
run([CHEMPROP, "predict", "--test-path", str(CP_DIR / "test.csv"), "--reaction-columns", "rxn",
     "--model-paths", str(CP_DIR / "model_random"), "--num-workers", "0",
     "--preds-path", str(CP_DIR / "test_preds.csv")], CP_DIR / "predict.log")

# Chemprop writes predictions into a column named after the target ("yield"),
# so take the true values from our own table (rows are in the same order).
y_pred_cp = pd.read_csv(CP_DIR / "test_preds.csv")["yield"].to_numpy()
y_true_cp = cp.loc[cp.split == "test", "yield"].to_numpy()
print(f"Chemprop  R2 = {r2_score(y_true_cp, y_pred_cp):.3f}   MAE = {mean_absolute_error(y_true_cp, y_pred_cp):.2f}")

### Your turn (Part 6)

1. Compare Chemprop with the random forests on the *same* test set. Is the neural network worth the extra cost and opacity here?
2. Make a `split` column that holds out one ligand (as in Part 5) and retrain Chemprop. Does a learned representation generalise better than fingerprints?
3. Our reaction SMILES are **not atom-mapped**, so the CGR cannot line up reactant and product atoms properly. Map them with [rxnmapper](https://github.com/rxn4chemistry/rxnmapper) (it needs its own environment) and see whether the out-of-sample performance improves.

---

### Take-home messages for the project

- On a full-factorial screen, almost any model interpolates well. **Generalising to new reagents or substrates is the hard part**, and you must test for it explicitly.
- For a small number of your own experiments (tens, not thousands), you cannot train a deep model from scratch. You need a method that works with little data, tells you how *uncertain* it is, and suggests which experiment to run next. That is **Bayesian optimisation**, in notebook 03.

In [ ]:
!pwd